# Redrob Ranker — Embedding Pre-computation

Run this on **GPU runtime** (Runtime → Change runtime type → T4 GPU).

Steps:
1. Upload `candidates.jsonl` to Google Drive
2. Run all cells
3. Download the 3 `.npy` files at the end
4. Place them in `redrob-ranker/embeddings/` on your local machine
5. Run `python rank.py --candidates ./candidates.jsonl --out ./submission.csv` locally

In [ ]:
# Step 1: Install dependencies
!pip install sentence-transformers -q

In [ ]:
# Step 2: Mount Google Drive (candidates.jsonl must be uploaded here)
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where you uploaded candidates.jsonl in your Drive
CANDIDATES_PATH = '/content/drive/MyDrive/candidates.jsonl'

In [ ]:
import json, gzip, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import os

OUT_DIR = Path('/content/embeddings')
OUT_DIR.mkdir(exist_ok=True)

# The JD query — what we're matching candidates against
JD_QUERY = """
Senior AI/ML Engineer role focused on production retrieval and ranking systems.

Requirements:
- Production experience with embeddings-based retrieval (sentence-transformers, BGE, E5, OpenAI embeddings)
- Vector database operations: Pinecone, Weaviate, Qdrant, FAISS, Milvus, Elasticsearch, OpenSearch
- Strong Python engineering with production deployments to real users at scale
- Evaluation frameworks for ranking: NDCG, MRR, MAP, A/B testing, offline evaluation
- 5-9 years experience at product companies (not pure IT services / consulting)

Nice to have:
- LLM fine-tuning: LoRA, QLoRA, PEFT
- Learning to rank: XGBoost, LightGBM, neural rankers
- RAG systems, distributed inference, MLflow

The ideal candidate has shipped at least one end-to-end ranking or recommendation system
to real users, has strong opinions on dense vs hybrid retrieval, and can evaluate
ranking systems rigorously. Product company experience preferred over IT services.
"""

def build_candidate_text(candidate):
    profile = candidate.get('profile', {})
    career = candidate.get('career_history', [])
    skills = candidate.get('skills', [])
    certifications = candidate.get('certifications', [])
    parts = []
    headline = profile.get('headline', '')
    summary = profile.get('summary', '')
    if headline: parts.append(headline)
    if summary: parts.append(summary)
    for role in career:
        title = role.get('title', '')
        company = role.get('company', '')
        desc = role.get('description', '')
        dur = role.get('duration_months', 0) or 0
        if desc:
            prefix = f'{title} at {company} ({dur}mo): ' if title else ''
            parts.append(prefix + desc)
    skill_strs = []
    for s in skills:
        name = s.get('name', '')
        prof = s.get('proficiency', '')
        dur = s.get('duration_months', 0) or 0
        if name:
            skill_strs.append(f'{name} ({prof}, {dur}mo)' if prof else name)
    if skill_strs: parts.append('Skills: ' + ', '.join(skill_strs))
    cert_names = [c.get('name', '') for c in certifications if c.get('name')]
    if cert_names: parts.append('Certifications: ' + ', '.join(cert_names))
    return ' | '.join(parts)

print('Loading model: all-MiniLM-L6-v2')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded!')

In [ ]:
# Embed JD
jd_emb = model.encode(JD_QUERY, normalize_embeddings=True)
np.save(OUT_DIR / 'jd_embedding.npy', jd_emb)
print(f'JD embedding saved. Shape: {jd_emb.shape}')

In [ ]:
# Stream and embed all candidates
candidate_ids = []
texts = []

open_fn = gzip.open if CANDIDATES_PATH.endswith('.gz') else open

with open_fn(CANDIDATES_PATH, 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc='Reading candidates', unit='cand'):
        line = line.strip()
        if not line: continue
        try:
            cand = json.loads(line)
            candidate_ids.append(cand.get('candidate_id', ''))
            texts.append(build_candidate_text(cand))
        except json.JSONDecodeError:
            continue

print(f'Total candidates: {len(texts)}')

In [ ]:
# Encode all — GPU makes this ~20x faster than CPU
print('Encoding on GPU...')
embeddings = model.encode(
    texts,
    batch_size=1024,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# Save to disk
np.save(OUT_DIR / 'candidate_embeddings.npy', embeddings.astype(np.float32))
np.save(OUT_DIR / 'candidate_ids.npy', np.array(candidate_ids, dtype=object))
print('Saved!')
print(f'  candidate_embeddings.npy  {embeddings.nbytes / 1e6:.1f} MB')
print(f'  candidate_ids.npy')
print(f'  jd_embedding.npy')

In [ ]:
# Download the files to your local machine
from google.colab import files
files.download(str(OUT_DIR / 'candidate_embeddings.npy'))
files.download(str(OUT_DIR / 'candidate_ids.npy'))
files.download(str(OUT_DIR / 'jd_embedding.npy'))
print('Downloads triggered!')

## After downloading
Move the 3 files into `redrob-ranker/embeddings/` on your local machine, then run:
```bash
python rank.py --candidates ./candidates.jsonl --out ./submission.csv
```